# AMD — análise financeira

Notebook para executar no **Google Colab**. Obtém preços históricos e demonstrações financeiras da AMD através de `yfinance`, cria gráficos e exporta os dados para CSV.

> Os dados dependem da disponibilidade do Yahoo Finance e podem ter atraso. Este notebook é informativo e não constitui aconselhamento financeiro.

In [ ]:
%pip -q install yfinance pandas matplotlib

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

TICKER = "AMD"
PERIODO = "5y"       # exemplos: 1y, 2y, 5y, max
INTERVALO = "1d"     # exemplos: 1d, 1wk, 1mo
PASTA = Path("dados_amd")
PASTA.mkdir(exist_ok=True)

amd = yf.Ticker(TICKER)
print(f"A obter dados de {TICKER}…")

## 1. Histórico de preços

In [ ]:
precos = amd.history(period=PERIODO, interval=INTERVALO, auto_adjust=False)
if precos.empty:
    raise RuntimeError("Sem preços disponíveis. Confirma a ligação e os parâmetros.")

display(precos.tail())
precos.to_csv(PASTA / "amd_historico.csv")

plt.figure(figsize=(13, 5))
plt.plot(precos.index, precos["Close"], label="Preço de fecho")
plt.title(f"{TICKER} — preço de fecho ({PERIODO})")
plt.xlabel("Data")
plt.ylabel("Preço (USD)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Volume negociado

In [ ]:
if "Volume" in precos.columns:
    plt.figure(figsize=(13, 4))
    plt.plot(precos.index, precos["Volume"])
    plt.title(f"{TICKER} — volume negociado ({PERIODO})")
    plt.xlabel("Data")
    plt.ylabel("Número de ações")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Demonstrações financeiras

In [ ]:
demonstracoes = {
    "resultados_anuais": amd.income_stmt,
    "resultados_trimestrais": amd.quarterly_income_stmt,
    "balanco_anual": amd.balance_sheet,
    "balanco_trimestral": amd.quarterly_balance_sheet,
    "fluxo_caixa_anual": amd.cashflow,
    "fluxo_caixa_trimestral": amd.quarterly_cashflow,
}

for nome, tabela in demonstracoes.items():
    if tabela is not None and not tabela.empty:
        tabela.to_csv(PASTA / f"amd_{nome}.csv")
        print(f"{nome}: {tabela.shape[0]} linhas × {tabela.shape[1]} períodos")
    else:
        print(f"{nome}: sem dados disponíveis")

display(amd.income_stmt)

## 4. Gráfico de receita e lucro líquido

In [ ]:
resultados = amd.income_stmt
if resultados is not None and not resultados.empty:
    linhas = {str(indice).lower(): indice for indice in resultados.index}
    receita_key = next((v for k, v in linhas.items() if "total revenue" in k or k == "revenue"), None)
    lucro_key = next((v for k, v in linhas.items() if "net income" in k), None)
    series = {}
    if receita_key is not None:
        series["Receita"] = pd.to_numeric(resultados.loc[receita_key], errors="coerce")
    if lucro_key is not None:
        series["Lucro líquido"] = pd.to_numeric(resultados.loc[lucro_key], errors="coerce")
    if series:
        financeiro = pd.DataFrame(series).sort_index()
        display(financeiro)
        financeiro.to_csv(PASTA / "amd_receita_lucro.csv")
        ax = financeiro.plot(kind="bar", figsize=(12, 5))
        ax.set_title(f"{TICKER} — receita e lucro líquido anual")
        ax.set_xlabel("Período")
        ax.set_ylabel("USD")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Não foi possível identificar receita/lucro nos dados disponíveis.")
else:
    print("Demonstração de resultados não disponível.")

## 5. Indicadores gerais da empresa

In [ ]:
try:
    info = amd.get_info()
    chaves = ["longName", "sector", "industry", "currency", "marketCap", "trailingPE", "forwardPE", "priceToBook", "profitMargins", "returnOnEquity", "dividendYield"]
    resumo = {chave: info.get(chave) for chave in chaves if info.get(chave) is not None}
    resumo_df = pd.DataFrame(resumo.items(), columns=["Indicador", "Valor"])
    display(resumo_df)
    resumo_df.to_csv(PASTA / "amd_resumo_empresa.csv", index=False)
except Exception as erro:
    print(f"Não foi possível obter os indicadores: {erro}")

## 6. Descarregar os ficheiros no computador

Executa a célula seguinte para descarregar o histórico de preços. Para descarregar todos os CSV, cria-se um ZIP.

In [ ]:
import zipfile
arquivo_zip = Path("dados_amd.zip")
with zipfile.ZipFile(arquivo_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for ficheiro in PASTA.glob("*"):
        zf.write(ficheiro, arcname=ficheiro.name)

print(f"Criado: {arquivo_zip.resolve()}")
from google.colab import files
files.download(str(arquivo_zip))